# Bayes by Backprop — Weight Uncertainty in Neural Networks (2015)

_Papers · Meta-learning, Bayesian & Robustness_

**Give every weight a learned mean and spread, and train the whole distribution by backprop.**

---

This notebook is a practice scaffold for the **Bayes by Backprop — Weight Uncertainty in Neural Networks (2015)** lesson. Run the cells, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np, matplotlib.pyplot as plt

## Reference implementation — PyTorch

In [ ]:
import torch, torch.nn as nn, torch.nn.functional as F, math, numpy as np
torch.manual_seed(0); np.random.seed(0)

# --- 0. Worked example: sample one weight + compute its KL term, by hand. ---
mu, rho, eps = torch.tensor(0.1), torch.tensor(-2.0), torch.tensor(0.5)
sigma = F.softplus(rho)                       # log(1+exp(rho))
w = mu + sigma*eps                            # reparameterization trick
s0 = 1.0                                       # prior std, prior = N(0,1)
kl = math.log(s0) - torch.log(sigma) + (sigma**2 + mu**2)/(2*s0**2) - 0.5
print("worked example: sigma=%.5f  w=%.5f  KL=%.4f" % (sigma.item(), w.item(), kl.item()))
# worked example: sigma=0.12693  w=0.16346  KL=1.5772


# --- 1. The NOVEL part: a Bayesian linear layer (mean + std per weight). ---
class BayesLinear(nn.Module):
    def __init__(self, n_in, n_out, prior_sigma=1.0):
        super().__init__()
        self.w_mu  = nn.Parameter(torch.empty(n_out, n_in).normal_(0, 0.1))
        self.w_rho = nn.Parameter(torch.full((n_out, n_in), -3.0))   # softplus(-3)~0.05
        self.b_mu  = nn.Parameter(torch.empty(n_out).normal_(0, 0.1))
        self.b_rho = nn.Parameter(torch.full((n_out,), -3.0))
        self.ps = prior_sigma
    def forward(self, x):
        w_sig, b_sig = F.softplus(self.w_rho), F.softplus(self.b_rho)
        w = self.w_mu + w_sig*torch.randn_like(w_sig)   # reparam: fresh noise each pass
        b = self.b_mu + b_sig*torch.randn_like(b_sig)
        return F.linear(x, w, b)
    def kl(self):                                         # closed-form Gaussian KL vs N(0, ps^2)
        def _k(mu, sig):
            return (math.log(self.ps) - torch.log(sig) + (sig**2 + mu**2)/(2*self.ps**2) - 0.5).sum()
        return _k(self.w_mu, F.softplus(self.w_rho)) + _k(self.b_mu, F.softplus(self.b_rho))

class BNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.l1, self.l2, self.l3 = BayesLinear(1,64), BayesLinear(64,64), BayesLinear(64,1)
    def forward(self, x):
        h = torch.relu(self.l1(x)); h = torch.relu(self.l2(h)); return self.l3(h)
    def kl(self): return self.l1.kl() + self.l2.kl() + self.l3.kl()

# --- 2. 1-D regression data in a CENTRAL BAND only (paper target, Sec 5.2). ---
def f(x): return x + 0.3*np.sin(2*np.pi*x) + 0.3*np.sin(4*np.pi*x)
N = 40
xt = np.random.uniform(-0.2, 0.6, (N,1)).astype(np.float32)
yt = (f(xt) + np.random.normal(0, 0.02, (N,1))).astype(np.float32)
X, Y = torch.tensor(xt), torch.tensor(yt)

# --- 3. Train on the ELBO / variational free energy (Eqn. 1). ---
net = BNN(); opt = torch.optim.Adam(net.parameters(), lr=0.01); noise = 0.02
for it in range(3000):
    opt.zero_grad()
    nll = sum(0.5*((net(X) - Y)**2).sum()/noise**2 for _ in range(2)) / 2   # likelihood cost (2 MC samples)
    loss = net.kl()/N + nll                                                  # complexity + likelihood
    loss.backward(); opt.step()

# --- 4. Predictive uncertainty = std of outputs across many weight samples. ---
def band(xs, S=200):
    xs = torch.tensor(xs.astype(np.float32))
    with torch.no_grad():
        samp = torch.stack([net(xs) for _ in range(S)], 0)
    return samp.mean(0).squeeze(-1).numpy(), samp.std(0).squeeze(-1).numpy()

grid = np.linspace(-1.5, 1.9, 60).reshape(-1,1)
m, s = band(grid)
in_mask  = (grid[:,0] > -0.2) & (grid[:,0] < 0.6)
far_mask = (grid[:,0] < -0.8) | (grid[:,0] > 1.2)
print("\nin-data   mean predictive std = %.4f" % s[in_mask].mean())
print("far-data  mean predictive std = %.4f" % s[far_mask].mean())
print("ratio far/in = %.1fx  (uncertainty grows away from the data)" % (s[far_mask].mean()/s[in_mask].mean()))

# --- 5. ABLATION: clamp every sigma -> 0 (point estimate). Uncertainty must vanish. ---
with torch.no_grad():
    for p in [net.l1.w_rho, net.l1.b_rho, net.l2.w_rho, net.l2.b_rho, net.l3.w_rho, net.l3.b_rho]:
        p.fill_(-30.0)
_, s0_ab = band(grid)
print("\nablation sigma->0: far-data mean std = %.6f  (overconfident: collapses)" % s0_ab[far_mask].mean())
# in-data   mean predictive std = 0.0149
# far-data  mean predictive std = 0.2595
# ratio far/in = 17.4x
# ablation sigma->0: far-data mean std = 0.000000
# (Our small run, not the paper's reported numbers. Exact values vary by seed/hardware.)

## Visualize the data & results

_After training on data that lives only in a central band, how does the network's predictive uncertainty (std across weight samples) behave as we move away from the data?_

In [ ]:
import torch, torch.nn as nn, torch.nn.functional as F, math, numpy as np
torch.manual_seed(0); np.random.seed(0)

class BayesLinear(nn.Module):
    def __init__(self, n_in, n_out, ps=1.0):
        super().__init__()
        self.w_mu  = nn.Parameter(torch.empty(n_out, n_in).normal_(0, 0.1))
        self.w_rho = nn.Parameter(torch.full((n_out, n_in), -3.0))
        self.b_mu  = nn.Parameter(torch.empty(n_out).normal_(0, 0.1))
        self.b_rho = nn.Parameter(torch.full((n_out,), -3.0)); self.ps = ps
    def forward(self, x):
        ws, bs = F.softplus(self.w_rho), F.softplus(self.b_rho)
        w = self.w_mu + ws*torch.randn_like(ws); b = self.b_mu + bs*torch.randn_like(bs)
        return F.linear(x, w, b)
    def kl(self):
        k = lambda mu, s: (math.log(self.ps) - torch.log(s) + (s**2+mu**2)/(2*self.ps**2) - 0.5).sum()
        return k(self.w_mu, F.softplus(self.w_rho)) + k(self.b_mu, F.softplus(self.b_rho))

class BNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.l1,self.l2,self.l3 = BayesLinear(1,64),BayesLinear(64,64),BayesLinear(64,1)
    def forward(self,x):
        h=torch.relu(self.l1(x)); h=torch.relu(self.l2(h)); return self.l3(h)
    def kl(self): return self.l1.kl()+self.l2.kl()+self.l3.kl()

def f(x): return x + 0.3*np.sin(2*np.pi*x) + 0.3*np.sin(4*np.pi*x)
N=40; xt=np.random.uniform(-0.2,0.6,(N,1)).astype(np.float32)
yt=(f(xt)+np.random.normal(0,0.02,(N,1))).astype(np.float32)
X,Y=torch.tensor(xt),torch.tensor(yt)

net=BNN(); opt=torch.optim.Adam(net.parameters(),lr=0.01); noise=0.02
for it in range(3000):
    opt.zero_grad()
    nll=sum(0.5*((net(X)-Y)**2).sum()/noise**2 for _ in range(2))/2
    (net.kl()/N + nll).backward(); opt.step()

grid=np.linspace(-1.5,1.9,60).reshape(-1,1)
with torch.no_grad():
    samp=torch.stack([net(torch.tensor(grid.astype(np.float32))) for _ in range(200)],0)
s=samp.std(0).squeeze(-1).numpy()
idx=np.linspace(0,59,16).astype(int)
print("predictive std (x, std):")
for i in idx: print("  [%.3f,%.4f]"%(grid[i,0],s[i]))
# climbs from ~0.55 at x=-1.5 down to ~0.007 in the data band up to ~0.22 at x=1.9.
# Our small run, not the paper's number.

## Practice

Try each one in the empty cell below it, then reveal the worked solution.

**Problem 1.** The uncertainty ablation. You have a trained Bayes-by-Backprop network whose predictive band
            widens far from the data. You now clamp every weight's raw spread to a very negative value
            (w_rho.fill_(-30.0)), so $\sigma = $ softplus(-30) $\approx 0$, and re-measure
            the predictive band. What did you just turn the network into, and what happens to the uncertainty far
            from the data?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Compute $\sigma$ after the clamp: softplus(-30) $\approx 9\times 10^{-14}$, effectively zero. — _Every weight collapses to a single fixed number $w = \mu + 0\cdot\epsilon = \mu$, so the distribution over weights becomes a point._
- Sample the net many times and take the per-point standard deviation of the outputs. — _With no spread in the weights, every forward pass produces the same output, so the cross-sample standard deviation is zero everywhere._
- Compare to before the clamp. — _The Bayesian net's band grew far from data; the clamped net's band is flat zero &mdash; an ordinary, overconfident network._

**Answer:** Clamping $\sigma \to 0$ turns each weight back into a single number &mdash; an ordinary
                 deterministic network. Every forward pass is now identical, so the predictive standard deviation is
                 zero everywhere, including far from the data where it should be large. In our run the
                 far-from-data band collapses from $\approx 0.26$ to $0.000000$. This is exactly the overconfident
                 extrapolation the paper contrasts against (&sect;5.2): the learned spreads $\sigma$ are what carry
                 the uncertainty, and removing them removes the network's ability to say "I do not know here."

</details>

**Problem 2.** Why does the predictive band stay narrow inside the data band but widen in the empty regions,
            even though all weights share the same prior?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Recall the two costs: likelihood (fit the data) and KL complexity (stay near the prior). — _Inside the data band, fitting the points requires the network's output there to be pinned down, which forces the relevant weight spreads $\sigma$ small. The likelihood term dominates._
- Reason about the empty regions: no data point constrains the output there. — _Only the prior speaks, so the KL term keeps the spreads $\sigma$ comparatively large; many different weight settings are equally consistent with the data._
- Translate weight spread into output spread: large $\sigma$ on the weights means sampled networks disagree most where they are least constrained. — _Predictive uncertainty is the spread of outputs across weight samples, so it grows exactly where the weights remain uncertain &mdash; far from data._

**Answer:** Because the data only constrains the network where the data is. In the central band the
                 likelihood cost forces the relevant weights to tight spreads, so sampled networks agree and the band
                 is narrow. In the empty regions no likelihood term pulls on the output, so the KL leaves those weight
                 spreads large and the sampled networks fan out &mdash; the band widens. Same prior everywhere; what
                 differs is how much data pins each region down. In our run the mean predictive standard deviation is
                 $\approx 0.015$ inside the data versus $\approx 0.26$ far away &mdash; about $17\times$ wider.

</details>

**Problem 3.** In the worked example you had $\mu = 0.1$, $\rho = -2.0$ (so $\sigma = 0.12693$), prior
            $\mathcal{N}(0,1)$, giving a KL of $\approx 1.577$. Suppose training instead drove this weight to a
            much larger spread, $\rho = +2.0$. Compute the new $\sigma$ and the new KL term. What does the change
            say about the complexity cost?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- New spread: $\sigma = \log(1+e^{2}) = \log(8.389) = 2.1269$. — _Softplus of a positive $\rho$ gives a spread well above the prior's $\sigma_0 = 1$._
- New KL $= \log\frac{1}{2.1269} + \frac{2.1269^2 + 0.1^2}{2} - \frac12 = (-0.7547) + 2.2675 - 0.5 = 1.0128. — _Now the $(\sigma^2)/2$ term grows and the $\log(\sigma_0/\sigma)$ term goes negative; the KL is still positive but smaller than before._
- Compare 1.577 (tight, $\sigma{=}0.127$) vs 1.013 (loose, $\sigma{=}2.127$). — _Both deviate from the prior, but a spread that is too tight is penalized more here than a spread somewhat larger than the prior &mdash; the KL is asymmetric in $\sigma$._

**Answer:** With $\rho = +2.0$: $\sigma = \log(1+e^{2}) = 2.1269$ and the KL term is
                 $\log\frac{1}{2.1269} + \frac{2.1269^2 + 0.01}{2} - 0.5 = 1.0128$. So loosening this weight from
                 $\sigma = 0.127$ to $\sigma = 2.13$ actually lowers its complexity cost (1.577 &rarr;
                 1.013) here, because a very tight spread is far from the prior's $\sigma_0 = 1$. The lesson: the KL
                 complexity cost wants each weight's spread to sit near the prior's, neither collapsed to zero
                 nor exploded &mdash; it is the likelihood term that earns the right to tighten $\sigma$ where data
                 demands it.

</details>